# Reproducing Isolation Forest results (Liu, Ting & Zhou, 2008)

This notebook runs Isolation Forest and LOF on public copies of the 12 datasets listed in the paper.
It compares AUC and runtime with Table 3.

All datasets are downloaded automatically from GitHub. Runtime depends on the computer.

**Scope:** this notebook reproduces the Isolation Forest and LOF comparisons, not the ORCA and RF columns.
It does not reproduce the additional experiments in Sections 5.2 to 5.4. Differences between the available data and the paper are listed below.

## Parameters

| Method | Parameter | Value | Source |
|---|---|---|---|
| Isolation Forest | `n_estimators` (number of trees, t) | 100 | paper, Section 4.1 |
| Isolation Forest | `max_samples` (sub-sampling size, ψ) | 256 | paper, Section 4.1 |
| Isolation Forest | `random_state` (random seed) | 0 to 9, so 10 runs per dataset | my choice; the paper does not say how many runs it used |
| Isolation Forest | `n_jobs` (CPU threads) | 1 | the paper used single-threaded jobs |
| Isolation Forest | all other parameters | scikit-learn defaults | |
| LOF | `n_neighbors` (k) | 10 | paper, Section 5 |
| LOF | `n_jobs` | 1 | |
| LOF | all other parameters | scikit-learn defaults (Euclidean distance) | |
| Both | anomaly score | `-score_samples(X)` for iForest, `-negative_outlier_factor_` for LOF | |
| Both | evaluation | ROC AUC; the labels are used only to compute the AUC | paper, Section 5 |
| Both | runtime | time for training and scoring in seconds, average of 10 runs | |
| Both | data | no scaling or other transformation by this notebook (except removing one attribute of Ionosphere and the Mulcross row ID); some downloaded files are already transformed, see "Notes on the data" | |

In [1]:
# Import libraries
import sys
import urllib.request
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import sklearn
from scipy.io import loadmat
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import LocalOutlierFactor

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)
print("NumPy", np.__version__, "| pandas", pd.__version__)


Python 3.13.16 | scikit-learn 1.6.1
NumPy 2.1.3 | pandas 2.2.3


## Step 1. Download the datasets

The paper uses 12 datasets in which the anomalies are known. The code downloads public copies of them:

* Http, ForestCover, Smtp, Shuttle, Mammography, Annthyroid, Satellite, Pima and Breastw: ADBench benchmark (GitHub)
* Arrhythmia and Ionosphere: ODDS collection (copy in the PyOD GitHub repository)
* Mulcross: CSV benchmark from the Online Isolation Forest repository (GitHub), using a fixed repository revision.

For every dataset, `X` contains the attributes (one row per record) and `y` the labels (1 = anomaly, 0 = normal).

In [2]:
# Where each dataset is downloaded from
adbench = "https://raw.githubusercontent.com/Minqi824/ADBench/main/adbench/datasets/Classical/"
odds = "https://raw.githubusercontent.com/yzhao062/pyod/master/notebooks/data/"

sources = {
    "Http": adbench + "16_http.npz",
    "ForestCover": adbench + "10_cover.npz",
    "Mulcross": "https://media.githubusercontent.com/media/ineveLoppiliF/Online-Isolation-Forest/e528c86b77219791d026d732f618efb0c7bde3c1/datasets/stationary/ifor/Mulcross.csv",
    "Smtp": adbench + "34_smtp.npz",
    "Shuttle": adbench + "32_shuttle.npz",
    "Mammography": adbench + "23_mammography.npz",
    "Annthyroid": adbench + "2_annthyroid.npz",
    "Satellite": adbench + "30_satellite.npz",
    "Pima": adbench + "29_Pima.npz",
    "Breastw": adbench + "4_breastw.npz",
    "Arrhythmia": odds + "arrhythmia.mat",
    "Ionosphere": odds + "ionosphere.mat",
}

In [3]:
# Download and load all datasets
data = {}

for name, source in sources.items():
    file_name = source.split("/")[-1]
    urllib.request.urlretrieve(source, file_name)

    if file_name.endswith(".csv"):
        # Mulcross: remove the row ID and separate the labels
        table = pd.read_csv(file_name)
        X = table.drop(columns=["id", "y"]).to_numpy(dtype=float)
        y = table["y"].to_numpy(dtype=int)
    elif file_name.endswith(".npz"):
        with np.load(file_name, allow_pickle=False) as loaded:
            X = loaded["X"].astype(float)
            y = loaded["y"].ravel().astype(int)
    else:
        loaded = loadmat(file_name)
        X = loaded["X"].astype(float)
        y = loaded["y"].ravel().astype(int)

    if name == "Ionosphere":
        X = X[:, 1:]  # remove the binary attribute from this ODDS copy

    assert X.shape[0] == len(y), name + ": data and labels differ in length"
    assert np.isfinite(X).all(), name + ": missing or infinite values"
    assert set(np.unique(y)) == {0, 1}, name + ": both label classes are required"
    data[name] = (X, y)
    print(name, "loaded:", X.shape[0], "records,", X.shape[1], "attributes,", y.sum(), "anomalies")

print("Datasets loaded:", len(data), "of", len(sources))

assert len(data) == 12, "Some datasets are missing"


Http loaded: 567498 records, 3 attributes, 2211 anomalies
ForestCover loaded: 286048 records, 10 attributes, 2747 anomalies
Mulcross loaded: 262144 records, 4 attributes, 26214 anomalies
Smtp loaded: 95156 records, 3 attributes, 30 anomalies
Shuttle loaded: 49097 records, 9 attributes, 3511 anomalies
Mammography loaded: 11183 records, 6 attributes, 260 anomalies
Annthyroid loaded: 7200 records, 6 attributes, 534 anomalies
Satellite loaded: 6435 records, 36 attributes, 2036 anomalies
Pima loaded: 768 records, 8 attributes, 268 anomalies
Breastw loaded: 683 records, 9 attributes, 239 anomalies
Arrhythmia loaded: 452 records, 274 attributes, 66 anomalies
Ionosphere loaded: 351 records, 32 attributes, 126 anomalies
Datasets loaded: 12 of 12


In [4]:
# Compare the datasets with Table 2 of the paper
paper_table2 = {  # (number of records n, number of attributes d, % anomalies)
    "Http": (567497, 3, 0.4),
    "ForestCover": (286048, 10, 0.9),
    "Mulcross": (262144, 4, 10),
    "Smtp": (95156, 3, 0.03),
    "Shuttle": (49097, 9, 7),
    "Mammography": (11183, 6, 2),
    "Annthyroid": (6832, 6, 7),
    "Satellite": (6435, 36, 32),
    "Pima": (768, 8, 35),
    "Breastw": (683, 9, 35),
    "Arrhythmia": (452, 274, 15),
    "Ionosphere": (351, 32, 36),
}

overview = []
for name, (X, y) in data.items():
    overview.append({
        "Dataset": name,
        "n (paper)": paper_table2[name][0],
        "n (mine)": X.shape[0],
        "d (paper)": paper_table2[name][1],
        "d (mine)": X.shape[1],
        "% anomalies (paper)": paper_table2[name][2],
        "% anomalies (mine)": round(100 * y.mean(), 2),
        "Duplicate rows": pd.DataFrame(X).duplicated().sum(),
    })
overview = pd.DataFrame(overview)
overview

,Dataset,n (paper),n (mine),d (paper),d (mine),% anomalies (paper),% anomalies (mine),Duplicate rows
0,Http,567497,567498,3,3,0.40,0.39,345598
1,ForestCover,286048,286048,10,10,0.90,0.96,0
2,Mulcross,262144,262144,4,4,10.00,10.00,0
3,Smtp,95156,95156,3,3,0.03,0.03,23926
4,Shuttle,49097,49097,9,9,7.00,7.15,0
5,Mammography,11183,11183,6,6,2.00,2.32,3335
6,Annthyroid,6832,7200,6,6,7.00,7.42,138
7,Satellite,6435,6435,36,36,32.00,31.64,0
8,Pima,768,768,8,8,35.00,34.90,0
9,Breastw,683,683,9,9,35.00,34.99,234


**Notes on the data**

* The table above shows where the datasets differ from Table 2 of the paper. For example, Annthyroid has
  7,200 records here and 6,832 in the paper, and Http has one record more.
* The Http and Smtp files contain log-transformed values, ln(x + 0.1).
* For Ionosphere, the first attribute is binary and was removed, because the paper uses 32 attributes.
* Some datasets contain duplicate rows (last column). For some of them scikit-learn warns that duplicates can make
  LOF results incorrect; Table A in Step 4 shows for which datasets this warning appeared.
* Mulcross is a public benchmark copy from the Online Isolation Forest repository. The code removes its row ID column and separates the existing binary labels; it does not generate new data.

## Step 2. Isolation Forest

For every dataset, the code trains Isolation Forest 10 times, each time with a different random seed (0 to 9).
In every run it measures the time for training and scoring and computes the AUC from the anomaly scores.
The results are the average AUC, the standard deviation (SD) of the AUC over the 10 runs, and the average time.

In [5]:
# Run Isolation Forest 10 times on every dataset
iforest_results = {}

for name, (X, y) in data.items():
    aucs = []
    times = []
    for seed in range(10):
        model = IsolationForest(n_estimators=100, max_samples=256, random_state=seed, n_jobs=1)
        start = perf_counter()
        model.fit(X)                      # build 100 trees, each on a random sample of 256 records
        scores = -model.score_samples(X)  # anomaly score: higher = more anomalous
        times.append(perf_counter() - start)
        aucs.append(roc_auc_score(y, scores))
    iforest_results[name] = (np.mean(aucs), np.std(aucs, ddof=1), np.mean(times))
    print(name, "AUC:", round(np.mean(aucs), 4), "| time (s):", round(np.mean(times), 4))

Http AUC: 0.9996 | time (s): 5.1938
ForestCover AUC: 0.8774 | time (s): 1.8043
Mulcross AUC: 0.9614 | time (s): 2.3235
Smtp AUC: 0.9043 | time (s): 0.9541
Shuttle AUC: 0.997 | time (s): 0.673
Mammography AUC: 0.8615 | time (s): 0.309
Annthyroid AUC: 0.8184 | time (s): 0.344
Satellite AUC: 0.7008 | time (s): 0.2511
Pima AUC: 0.6707 | time (s): 0.196
Breastw AUC: 0.9873 | time (s): 0.1968
Arrhythmia AUC: 0.809 | time (s): 0.1896
Ionosphere AUC: 0.8461 | time (s): 0.2979


## Step 3. LOF

LOF has no randomness, so the AUC is the same in every run. The code still runs it 10 times per dataset
to get an average runtime, like for Isolation Forest. It also records whether scikit-learn warned
that duplicate values can make the LOF result incorrect.

In [6]:
# Run LOF 10 times on every dataset
lof_results = {}

for name, (X, y) in data.items():
    times = []
    warned = False
    for run in range(10):
        lof = LocalOutlierFactor(n_neighbors=10, n_jobs=1)
        with warnings.catch_warnings(record=True) as caught:  # collect warnings instead of printing them
            warnings.simplefilter("always")
            start = perf_counter()
            lof.fit_predict(X)
            scores = -lof.negative_outlier_factor_  # LOF value: higher = more anomalous
            times.append(perf_counter() - start)
        if any("Duplicate values" in str(w.message) for w in caught):
            warned = True
        if run == 0:
            for warning in caught:
                if "Duplicate values" not in str(warning.message):
                    print(name, str(warning.message))
    auc = roc_auc_score(y, scores)
    lof_results[name] = (auc, np.mean(times), warned)
    print(name, "AUC:", round(auc, 4), "| time (s):", round(np.mean(times), 4), "| duplicate warning:", warned)

Http AUC: 0.3616 | time (s): 16.0378 | duplicate warning: True
ForestCover datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
ForestCover AUC: 0.5668 | time (s): 7.7246 | duplicate warning: False
Mulcross datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
Mulcross AUC: 0.5848 | time (s): 9.0435 | duplicate warning: False
Smtp datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
Smtp AUC: 0.4908 | time (s): 1.44 | duplicate warning: True
Shuttle datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.da

## Step 4. Comparison with the paper (Table 3)

Table A compares the AUC, Table B the runtime. "Difference" = my result minus the paper.
"LOF duplicate warning" shows whether scikit-learn warned that duplicate values can make the LOF result incorrect.
The paper reports AUC with two decimals. Empty values (NaN / None) mean that the paper has no value ("NA").
The runtimes come from different computers: the paper reports CPU time on 2008 hardware,
this notebook measures elapsed time.

In [7]:
# Values from Table 3 of the paper: AUC iForest, AUC LOF, time iForest (s), time LOF (s); None = NA in the paper
paper_table3 = {
    "Http": (1.00, None, 15.58, None),
    "ForestCover": (0.88, None, 16.33, None),
    "Mulcross": (0.97, None, 12.52, None),
    "Smtp": (0.88, None, 2.72, None),
    "Shuttle": (1.00, 0.55, 3.13, 7489.74),
    "Mammography": (0.86, 0.67, 0.66, 14647.00),
    "Annthyroid": (0.82, 0.72, 0.51, 72.02),
    "Satellite": (0.71, 0.52, 1.63, 217.39),
    "Pima": (0.67, 0.49, 0.28, 1.14),
    "Breastw": (0.99, 0.37, 0.28, 1.77),
    "Arrhythmia": (0.80, 0.73, 2.98, 6.35),
    "Ionosphere": (0.85, 0.89, 0.48, 0.64),
}

In [8]:
# Table A: AUC, paper vs. my results
auc_table = []
for name in data:
    paper_if_auc, paper_lof_auc, paper_if_time, paper_lof_time = paper_table3[name]
    my_if_auc, my_if_sd, my_if_time = iforest_results[name]
    my_lof_auc, my_lof_time, my_lof_warning = lof_results[name]
    auc_table.append({
        "Dataset": name,
        "iForest AUC (paper)": paper_if_auc,
        "iForest AUC (mine)": round(my_if_auc, 3),
        "iForest SD (mine)": round(my_if_sd, 3),
        "iForest difference": round(my_if_auc - paper_if_auc, 3) + 0.0,  # + 0.0 avoids showing -0.0
        "LOF AUC (paper)": paper_lof_auc,
        "LOF AUC (mine)": round(my_lof_auc, 3),
        "LOF difference": None if paper_lof_auc is None else round(my_lof_auc - paper_lof_auc, 3) + 0.0,
        "LOF duplicate warning": "yes" if my_lof_warning else "no",
    })
auc_table = pd.DataFrame(auc_table)
auc_table

,Dataset,iForest AUC (paper),iForest AUC (mine),iForest SD (mine),iForest difference,LOF AUC (paper),LOF AUC (mine),LOF difference,LOF duplicate warning
0,Http,1.00,1.000,0.000,0.000,NaN,0.362,NaN,yes
1,ForestCover,0.88,0.877,0.037,-0.003,NaN,0.567,NaN,no
2,Mulcross,0.97,0.961,0.006,-0.009,NaN,0.585,NaN,no
3,Smtp,0.88,0.904,0.006,0.024,NaN,0.491,NaN,yes
4,Shuttle,1.00,0.997,0.001,-0.003,0.55,0.552,0.002,no
5,Mammography,0.86,0.861,0.008,0.001,0.67,0.671,0.001,yes
6,Annthyroid,0.82,0.818,0.017,-0.002,0.72,0.725,0.005,no
7,Satellite,0.71,0.701,0.020,-0.009,0.52,0.522,0.002,no
8,Pima,0.67,0.671,0.007,0.001,0.49,0.494,0.004,no
9,Breastw,0.99,0.987,0.001,-0.003,0.37,0.439,0.069,yes


In [9]:
# Table B: runtime in seconds, paper vs. my results
time_table = []
for name in data:
    paper_if_auc, paper_lof_auc, paper_if_time, paper_lof_time = paper_table3[name]
    my_if_auc, my_if_sd, my_if_time = iforest_results[name]
    my_lof_auc, my_lof_time, my_lof_warning = lof_results[name]
    time_table.append({
        "Dataset": name,
        "iForest time (paper)": paper_if_time,
        "iForest time (mine)": round(my_if_time, 3),
        "LOF time (paper)": paper_lof_time,
        "LOF time (mine)": round(my_lof_time, 3),
    })
time_table = pd.DataFrame(time_table)
time_table

,Dataset,iForest time (paper),iForest time (mine),LOF time (paper),LOF time (mine)
0,Http,15.58,5.194,NaN,16.038
1,ForestCover,16.33,1.804,NaN,7.725
2,Mulcross,12.52,2.323,NaN,9.043
3,Smtp,2.72,0.954,NaN,1.440
4,Shuttle,3.13,0.673,7489.74,5.760
5,Mammography,0.66,0.309,14647.00,0.395
6,Annthyroid,0.51,0.344,72.02,0.115
7,Satellite,1.63,0.251,217.39,0.334
8,Pima,0.28,0.196,1.14,0.006
9,Breastw,0.28,0.197,1.77,0.010


## Step 5. Save the results
Run this cell to save the three tables as CSV files. Download them from the Files panel in Colab.

In [ ]:
# Save the main results
overview.to_csv("table_datasets.csv", index=False)
auc_table.to_csv("table_auc.csv", index=False)
time_table.to_csv("table_runtime.csv", index=False)
print("Saved: table_datasets.csv, table_auc.csv, table_runtime.csv")
